In [ ]:
#this block is for running a command on the RFsoC assuming you are 
#running one of the jupyter notebooks on the device but from a separate computer/visual studio code:
import os
dir = os.getcwd()
print(dir)
print(os.listdir())
# print(os.getcwd())

# os.system("pip install gitpython")#this will pull from the github to make sure you have the most current updates

import git  # pip install gitpython
g = git.cmd.Git(dir)
try:
    g.pull()
except:
    if 'amo_qick' in os.listdir():
        os.chdir('amo_qick')
        g.pull

from qick import *
from qick.asm_v2 import *
from qick.tprocv2_assembler import *
%matplotlib inline
import matplotlib.pyplot as plt

#This line is to sync to an external clock which needs to be 10 Mhz
#this line currently loads tproc1 firmware and is just to make sure everything is loaded properly
#soc = QickSoc(external_clk=True)

dir = os.getcwd()
print(dir)
#this line downloads the new hardware (just make sure thats what you want)
#soc = QickSoc()
#soc = QickSoc(bitfile =f'{dir}/tests/d_1.bit', download=True)
soc = QickSoc(bitfile =f'{dir}/tests/rf_board_firmware/ADC_0.bit', download=True)
soccfg = soc
print(soccfg)


/home/xilinx/jupyter_notebooks/amo_qick
['README.md', 'qick_demos', 'qick_lib', 'docs', 'LICENSE', 'pyro4', '.git', '.github', '.readthedocs.yaml', 'hardware', 'graphics', 'aws', 'tests', '.gitignore', 'setup.py', '.ipynb_checkpoints', 'CITATION.cff', 'firmware']
/home/xilinx/jupyter_notebooks/amo_qick
QICK running on ZCU216, software version 0.2.388

Firmware configuration (built Fri Feb 20 13:25:05 2026):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 1, ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 65536 complex samples (6.838 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, ra

In [31]:
from qick.asm_v2 import AsmV2
class PulseUpdateProgram2(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1, mixer_freq=0)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch, phase=cfg['ro_phase'])


        pulse_len = cfg['pulse_len']
        self.add_gauss(ch=gen_ch, name="gauss", sigma=pulse_len/10, length=pulse_len)
        
        self.add_pulse(ch=gen_ch, name="x/2", ro_ch=ro_ch, 
                       style="arb", 
                       envelope="gauss", 
                       freq=cfg['freq'], 
                       phase=0,
                       gain=1.0, 
                      )
        self.add_pulse(ch=gen_ch, name="y/2", ro_ch=ro_ch, 
                       style="arb", 
                       envelope="gauss", 
                       freq=cfg['freq'], 
                       phase=90,
                       gain=1.0, 
                      )

        sub = AsmV2()
        sub.pulse(ch=cfg['gen_ch'], name="x/2", t=0)
        sub.delay(pulse_len)
        self.add_subroutine("play_x/2", sub)

        sub = AsmV2()
        sub.pulse(ch=cfg['gen_ch'], name="y/2", t=0)
        sub.delay(pulse_len)
        self.add_subroutine("play_y/2", sub)

        sub = AsmV2()
        for wname in self.list_pulse_waveforms("x/2")+self.list_pulse_waveforms("y/2"):
            sub.read_wmem(name=wname)
            sub.inc_reg(dst='w_phase', src=self.deg2reg(180, gen_ch=cfg['gen_ch']))
            sub.write_wmem(name=wname)
        self.add_subroutine("virt_z", sub)

        self.add_reg('myIQ')
        sub = AsmV2()
        sub.read_input(cfg['ro_ch'])
        sub.append_macro(AsmInst(inst={'CMD':"REG_WR", 'DST':'&0', 'ADDR':cfg['read_out_reg']}, addr_inc=1))
        #sub.read_dmem("myIQ", cfg['read_out_reg'])

        self.add_subroutine("read_I/Q", sub)

        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)
    
    
    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        
        self.call("read_I/Q")       

        self.call("play_x/2")
        self.call("play_y/2")
        self.call("play_x/2")
        self.call("play_y/2")
        self.call("virt_z")
        self.call("play_x/2")
        self.call("play_y/2")
        self.call("play_x/2")
        self.call("play_y/2")

        # reset the phase to 0
        for wname in self.list_pulse_waveforms("x/2"):
            self.read_wmem(name=wname)
            self.write_reg(dst='w_phase', src=0)
            self.write_wmem(name=wname)
        for wname in self.list_pulse_waveforms("y/2"):
            self.read_wmem(name=wname)
            self.write_reg(dst='w_phase', src=self.deg2reg(90, gen_ch=cfg['gen_ch']))
            self.write_wmem(name=wname)
        
config = {'gen_ch': 1,
          'ro_ch': 10,
          'freq': 100,
          'trig_time': 0.4,
          'ro_len': 1.0,
          'pulse_len': 0.1,
          'ro_phase': -0,
          'read_out_reg': str(soccfg['readouts'][10]['tproc_ch'])
         }

prog = PulseUpdateProgram2(soccfg, reps=1, final_delay=0.5, cfg=config)
print(prog)
iq_list = prog.acquire_decimated(soc)
t = prog.get_time_axis(ro_index=0)

fig = plt.figure(figsize=(12,3))

iq = iq_list[0]
plt.plot(t, iq[:,0], label="I value")
plt.plot(t, iq[:,1], label="Q value")
# plt.plot(t, np.abs(iq.dot([1,1j])), label="magnitude")
plt.legend()
plt.ylabel("a.u.")
plt.xlabel("us");

generator 1 doesn't have a digital mixer, but mixer_freq was defined


KeyError: 'SRC'